##Creating a table of relevant MSDS variables linked to CCU063 maternity interpreter cohort 

Purpose - to link MSDS previous early loss with maternity interpreter cohort

Authors - Majel McGranahan supported by Lars Murdock

Reviewed - Not reviewed, needs cleaning up by MM

#0 Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

# 1 Load Maternity Interpreter Cohort Table

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_previousstillbirths')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (maternity_interpreter_cohort
.select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'person_id_demo', 'Dob', 'eth5', 'region', 'imd_quintile', 'imd_decile', 'in_gdppr', 'gdppr_min_date', 'interpreter_use', 'record_before_lookback', 'ageatbookingmother', 'delivery_date', 'agefinal', 'folicacid', 'ovsvischcat', 'ovsvischcatappdate', 'complexsocialfactors', 'gestagebooking', 'previouslivebirths', 'previousstillbirths')
        )



# 2 Load MSDS table



In [0]:
msds_demo= (spark.table(f'{dbc_old}.msds_v2_demographics_booking_and_pregnancy_all_years_archive')
          #.filter(F.col('ADMIDATE') > "2018-01-01")
          .filter(f.col('archived_on') == tmp_archived_on)
          )

In [0]:
msds_demo.count()

In [0]:
msds_demo= (
    msds_demo
    .select(f.col('person_id_mother_deid').alias('person_id_mother_msds'), 
            f.col('uniqpregid').alias('uniquepregid_msds'),
           # f.col('ovsvischcat').alias('ovsvischcat'),
           # f.col('ovsvischcatappdate').alias('ovsvischcatappdate'),
           #f.col('langcode').alias('langcode'),
           #f.col('complexsocialfactorsind').alias('complexsocialfactors'),
           #f.col('previouslivebirths').alias('previouslivebirths'),
           #f.col('previousstillbirths').alias('previousstillbirths'),
           f.col('previouslosseslessthan24weeks').alias('previouslosseslessthan24weeks'),
          #f.col('folicacidsupplement').alias('folicacid1'),
          #f.col('gestagebooking').alias('gestagebooking'),
          #f.col('archived_on').alias('archived_on'),
          #           .distinct() 
           ).dropDuplicates()
)

In [0]:
msds_demo.count()

In [0]:
display(msds_demo)

In [0]:
msds_demo_previouslosseslessthan24weeks =    (
msds_demo
.select(f.col('person_id_mother_msds').alias('person_id_mother_msds'), 
            f.col('uniquepregid_msds').alias('uniquepregid_msds'),
            #f.col('archived_on').alias('archived_on'),
          f.col('previouslosseslessthan24weeks').alias('previouslosseslessthan24weeks')))



#3 Clean previous live births variable

In [0]:
#Code from CCU018_02_D03-cohort msds

msds_demo_previouslosseslessthan24weeks = (msds_demo_previouslosseslessthan24weeks
#        .select("uniqpregid" ,  "antenatalappdate"  , "previouscaesareansections")
        .where(f.col("previouslosseslessthan24weeks").isNotNull())
        .dropDuplicates(["uniquepregid_msds", "previouslosseslessthan24weeks" ])
        # dropping anomalous large numbers
        .where( ( f.col("previouslosseslessthan24weeks") >= min_earlyloss ) & ( f.col("previouslosseslessthan24weeks") <= max_earlyloss ) )
        # logic model such that highest number in pregnancy is prioritised
        .sort("uniquepregid_msds", "previouslosseslessthan24weeks", ascending= False)
        .dropDuplicates(["uniquepregid_msds"])
)


#if checks_on:
display(msds_demo_previouslosseslessthan24weeks)



In [0]:
tab(msds_demo_previouslosseslessthan24weeks, 'previouslosseslessthan24weeks')

In [0]:
##check
count_var(msds_demo_previouslosseslessthan24weeks, 'person_id_mother_msds')

In [0]:
##check
count_var(msds_demo_previouslosseslessthan24weeks, 'uniquepregid_msds')

In [0]:
display(msds_demo_previouslosseslessthan24weeks)

#4 Join MSDS to Maternity interpreter cohort table 

In [0]:
##Attempting left join based on https://www.geeksforgeeks.org/pyspark-join-types-join-two-dataframes/

##check row count in each table before and after join (row count in output table will be same as left table row count - filtered lookup)

# left join on two dataframes 
maternity_interpreter_previouslosseslessthan24weeks=maternity_interpreter_cohort.join(msds_demo_previouslosseslessthan24weeks, 
               maternity_interpreter_cohort.uniqpregid == msds_demo_previouslosseslessthan24weeks.uniquepregid_msds,  
               "left")




#display table
display(maternity_interpreter_previouslosseslessthan24weeks)
display(maternity_interpreter_previouslosseslessthan24weeks.printSchema())

In [0]:
count_var(maternity_interpreter_previouslosseslessthan24weeks, 'person_id_mother_deid')

# 5 Add previous loss binary variable

In [0]:
maternity_interpreter_previouslosseslessthan24weeks=maternity_interpreter_previouslosseslessthan24weeks.withColumn('prev_loss', 
                                                                       f.when(f.col('previouslosseslessthan24weeks') < 1, 'no')
                                                                       .when(f.col('previouslosseslessthan24weeks') >= 1, 'yes')
                                                                       .otherwise(None))

In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'prev_loss')

#6. Add parity variable

In [0]:
#from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Add new column 'parity'
maternity_interpreter_previouslosseslessthan24weeks = maternity_interpreter_previouslosseslessthan24weeks.withColumn("parity", col("previouslivebirths") + col("previousstillbirths"))



In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'parity')

# 7. Add nulliparous binary variable

In [0]:
maternity_interpreter_previouslosseslessthan24weeks=maternity_interpreter_previouslosseslessthan24weeks.withColumn('nulliparous', 
                                                                       f.when((f.col('previousstillbirths') < 1) & (f.col('previouslivebirths') < 1), 'yes')
                                                                       .when((f.col('previousstillbirths') >= 1) | (f.col('previouslivebirths') >= 1), 'no')
                                                                       .otherwise(None))

In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'nulliparous')

#8. Add previous pregnancy binary variable

In [0]:
maternity_interpreter_previouslosseslessthan24weeks=maternity_interpreter_previouslosseslessthan24weeks.withColumn('prev_preg', 
                                                                       f.when((f.col('previousstillbirths') < 1) & (f.col('previouslivebirths') < 1) & (f.col('previouslosseslessthan24weeks') < 1), 'no')
                                                                       .when((f.col('previousstillbirths') >= 1) | (f.col('previouslivebirths') >= 1) | (f.col('previouslosseslessthan24weeks') >= 1), 'yes')
                                                                       .otherwise(None))

In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'prev_preg')

#9. Add gestational age at booking in weeks

In [0]:
maternity_interpreter_previouslosseslessthan24weeks = maternity_interpreter_previouslosseslessthan24weeks.withColumn("gestagebookingweeks", col("gestagebooking") / 7)

In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'gestagebookingweeks')

#10. Add gestational age at booking above 10 weeks variable

In [0]:
maternity_interpreter_previouslosseslessthan24weeks=maternity_interpreter_previouslosseslessthan24weeks.withColumn('booking_after_10weeks', 
                                                                       f.when((f.col('gestagebookingweeks') > 10.1), 'yes')
                                                                       .when((f.col('gestagebookingweeks') <= 10.1), 'no')
                                                                       .otherwise(None))

In [0]:
tab(maternity_interpreter_previouslosseslessthan24weeks, 'booking_after_10weeks')

#11 Save table with previouslosseslessthan24weeks

In [0]:
outName = f'{proj}_maternity_interpreter_previouslosseslessthan24weeks'

# save
maternity_interpreter_previouslosseslessthan24weeks.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
maternity_interpreter_previouslosseslessthan24weeks = spark.table(f'{dbc}.{proj}_maternity_interpreter_previouslosseslessthan24weeks')